# Exploratory Data Analysis (EDA)

Before modelling, you *look* at the data. EDA is the first real use of
[`polars`](../01-foundations/polars-basics.ipynb) beyond syntax: loading a messy
dataset, summarising it, and spotting the problems the
[ETL chapter](../01c-etl/data-preparation.ipynb) will then fix.

We use a deliberately messy `customers.csv` (churn data with missing values,
an outlier, inconsistent text, and class imbalance) — the same dataset the ETL
and Capstone chapters build on. This builds on
[Foundations](../01-foundations/polars-basics.ipynb).

In [ ]:
:dep polars = { version = "0.44", features = ["lazy", "ndarray", "parquet", "strings"] }
use polars::prelude::*;

let df = CsvReadOptions::default()
    .with_has_header(true)
    .try_into_reader_with_file_path(Some("/book/data/customers.csv".into()))?
    .finish()?;
println!("shape = {:?}", df.shape());
println!("{}", df.head(Some(5)));

## Structural summary

Column types (schema) and how many values are missing per column:

In [ ]:
println!("schema:\n{:?}", df.schema());
println!("nulls per column:\n{}", df.null_count());

`age` and `income` have missing values. Now summary statistics for the numeric
columns (a hand-rolled `describe()` — mean, median, spread, range):

In [ ]:
let stats = df.clone().lazy().select([
    col("income").mean().alias("mean"),
    col("income").median().alias("median"),
    col("income").std(1).alias("std"),
    col("income").min().alias("min"),
    col("income").max().alias("max"),
]).collect()?;
println!("income summary:\n{}", stats);

Note how far `max` sits above the `median` — a strong hint of an **outlier**
(one customer has a ~900k income). We'll confirm that below.

## Class balance

For a classification target, check how balanced the classes are — imbalance
affects both modelling and evaluation:

In [ ]:
let balance = df.clone().lazy()
    .group_by([col("churned")])
    .agg([len().alias("count")])
    .sort(["churned"], Default::default())
    .collect()?;
println!("class balance:\n{}", balance);

Roughly 4:1 in favour of non-churners — a **mild imbalance** the
[ETL](../01c-etl/data-preparation.ipynb) and [Model
Evaluation](../01d-evaluation/cross-validation.ipynb) chapters will keep in mind.

## Distribution of a feature

A histogram of `income` (excluding the outlier for readability), built by
extracting the column to an `ndarray` and binning it by hand:

In [ ]:
:dep plotters = { version = "0.3", default-features = false, features = ["evcxr", "all_series", "all_elements"] }
use plotters::prelude::*;

// Pull non-null incomes below 200k into a Vec<f64> via the ndarray bridge.
let incomes: Vec<f64> = {
    let m = df.clone().lazy()
        .filter(col("income").is_not_null().and(col("income").lt(lit(200000))))
        .select([col("income")])
        .collect()?
        .to_ndarray::<Float64Type>(IndexOrder::C)?;
    (0..m.nrows()).map(|i| m[[i, 0]]).collect()
};

// 6 bins from 20k to 130k.
let (lo, hi, nbins) = (20000.0_f64, 130000.0_f64, 6usize);
let width = (hi - lo) / nbins as f64;
let mut counts = vec![0u32; nbins];
for v in &incomes {
    let mut b = ((v - lo) / width) as usize;
    if b >= nbins { b = nbins - 1; }
    counts[b] += 1;
}
let max_count = *counts.iter().max().unwrap_or(&1);

evcxr_figure((480, 300), |root| {
    root.fill(&WHITE)?;
    let mut chart = ChartBuilder::on(&root)
        .caption("Income distribution", ("sans-serif", 18))
        .margin(10)
        .x_label_area_size(35)
        .y_label_area_size(35)
        .build_cartesian_2d(lo..hi, 0u32..(max_count + 1))?;
    chart.configure_mesh().x_desc("income").y_desc("count").draw()?;
    chart.draw_series(counts.iter().enumerate().map(|(i, &c)| {
        let x0 = lo + i as f64 * width;
        Rectangle::new([(x0, 0), (x0 + width * 0.9, c)], BLUE.filled())
    }))?;
    Ok(())
})

## Outlier flagging (IQR rule)

The classic rule: flag values outside `[Q1 − 1.5·IQR, Q3 + 1.5·IQR]`. We compute
the quartiles from the sorted incomes. **Flagging is not removing** — what to do
about the outlier is an [ETL](../01c-etl/data-preparation.ipynb) decision.

In [ ]:
// Everything is wrapped in a block so the closure `q` stays local — evcxr can't
// persist a closure's type across cells (see the crate reference appendix).
{
    let mut all: Vec<f64> = {
        let m = df.clone().lazy()
            .filter(col("income").is_not_null())
            .select([col("income")])
            .collect()?
            .to_ndarray::<Float64Type>(IndexOrder::C)?;
        (0..m.nrows()).map(|i| m[[i, 0]]).collect()
    };
    all.sort_by(|a, b| a.partial_cmp(b).unwrap());
    let q = |p: f64| all[((all.len() as f64 - 1.0) * p).round() as usize];
    let (q1, q3) = (q(0.25), q(0.75));
    let iqr = q3 - q1;
    let upper = q3 + 1.5 * iqr;
    let flagged: Vec<f64> = all.iter().copied().filter(|v| *v > upper).collect();
    println!("Q1={} Q3={} IQR={}", q1, q3, iqr);
    println!("upper fence = {}", upper);
    println!("flagged outliers (> fence): {:?}", flagged);
}

## Data quality summary

The output of EDA is this short list of problems, which the ETL chapter opens by
addressing point-by-point:

1. **Missing values** in `age` and `income` → decide drop vs. impute.
2. **Outlier** income (~900k) → decide cap vs. remove.
3. **Inconsistent text** in `city` (`"berlin"`, `"Rome "`) → normalise case/whitespace.
4. **A duplicate-looking row** → deduplicate.
5. **Mild class imbalance** (~4:1) → keep in mind for evaluation.

Next: [ETL & Data Preparation](../01c-etl/data-preparation.ipynb), which turns
this list into a clean, model-ready dataset.